# Purpose of this notebook:

This notebook is dedicated to determining a baseline for the ML models for the thesis: "Prognostic Value of Baseline and Pre-Lymphodepletion PET Imaging in DLBCL Patients Undergoing CAR T-Cell Therapy".
The workflow is as below:

- 4 feature sets are trained and evaluated (plus one big dataset including all the features for comprehensiveness). This is so we can measure the performance changes as we go from clinical features only, to single point radiomic features + clinical features, to delta radiomic features + clinical features. 
- 3 Models: Logistic Regression, Decision Tree Classifier and Random Forest Classifier were used. The rationale behind the selection of these models is explained in the thesis.
- The performance is calculated using cross-validation with RepeatedStratifiedKFold (implemented within the model_eval script) and 4 metrics were measured to obtain a comprehenisve view of the model's performance: ROC-AUC, Precision, Recall, F1-score.

## Result:

- All models performed nearly naively and are extremely overfit due to having a wide dataset (features >> samples).
- Toxicity CRS>=1 was excluded from further analysis due to heavy imbalance (only 6 patients didn't experience this toxicity).

# Loading and Preprocessing

In [1]:
from data_loader import load_data

In [2]:
patients,\
X_clinical,\
X_clinical_A,\
X_clinical_B,\
X_clinical_Delta,\
targets = load_data()

In [3]:
datasets = {
    "Patients": patients,
    "Clinical": X_clinical,
    "Clinical_A": X_clinical_A,
    "Clinical_B": X_clinical_B,
    "Clinical_Delta": X_clinical_Delta
}

In [4]:
cat_cols = datasets['Clinical'].select_dtypes(exclude='number').columns.tolist()

In [5]:
# Using ColumnTransformer to apply different transformations to numeric and categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Create a ColumnTransformer to apply StandardScaler to numeric columns and leave categorical columns unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', 'passthrough', cat_cols)],
        remainder= StandardScaler()
    
)



In [6]:
from model_eval import result_viewer

# ICANS 0,1

In [7]:
y = targets['ae_summ_icans_v2']

In [8]:
y.value_counts()

ae_summ_icans_v2
1.0    48
0.0    37
Name: count, dtype: int64

In [9]:
icans_1 = result_viewer(datasets, y, preprocessor)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5189 ± 0.1175,0.6060 ± 0.1477,0.5852 ± 0.1454,0.5899 ± 0.1059,0.5251 ± 0.1508
train_roc_auc,1.0000 ± 0.0000,0.9173 ± 0.0251,0.9993 ± 0.0011,0.9993 ± 0.0015,1.0000 ± 0.0002
test_precision,0.5977 ± 0.0950,0.6811 ± 0.1266,0.6419 ± 0.1277,0.6531 ± 0.0913,0.6104 ± 0.1183
train_precision,1.0000 ± 0.0000,0.8487 ± 0.0322,0.9849 ± 0.0184,0.9847 ± 0.0148,0.9962 ± 0.0091
test_recall,0.6444 ± 0.1423,0.7267 ± 0.1242,0.6656 ± 0.1455,0.6817 ± 0.1644,0.6517 ± 0.1620
train_recall,1.0000 ± 0.0000,0.8918 ± 0.0358,0.9935 ± 0.0113,0.9961 ± 0.0093,0.9921 ± 0.0120
test_f1,0.6120 ± 0.0975,0.6924 ± 0.0889,0.6435 ± 0.1071,0.6591 ± 0.1169,0.6208 ± 0.1219
train_f1,1.0000 ± 0.0000,0.8691 ± 0.0254,0.9890 ± 0.0102,0.9903 ± 0.0099,0.9941 ± 0.0077


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5287 ± 0.1325,0.5749 ± 0.1145,0.5215 ± 0.1227,0.5339 ± 0.1192,0.4773 ± 0.1237
train_roc_auc,0.9270 ± 0.0251,0.9096 ± 0.0261,0.9229 ± 0.0253,0.9260 ± 0.0326,0.9259 ± 0.0227
test_precision,0.5864 ± 0.1328,0.6131 ± 0.1574,0.5947 ± 0.1083,0.5923 ± 0.1153,0.5656 ± 0.1042
train_precision,0.8875 ± 0.0445,0.8643 ± 0.0632,0.8885 ± 0.0548,0.8624 ± 0.0497,0.8905 ± 0.0483
test_recall,0.6167 ± 0.2011,0.6117 ± 0.2211,0.6489 ± 0.1745,0.5956 ± 0.2171,0.6039 ± 0.1803
train_recall,0.9371 ± 0.0666,0.8741 ± 0.0908,0.8944 ± 0.0914,0.9506 ± 0.0656,0.9152 ± 0.0737
test_f1,0.5868 ± 0.1390,0.5947 ± 0.1651,0.6071 ± 0.1109,0.5794 ± 0.1593,0.5705 ± 0.1149
train_f1,0.9091 ± 0.0310,0.8629 ± 0.0367,0.8858 ± 0.0347,0.9013 ± 0.0277,0.8993 ± 0.0312


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5329 ± 0.1572,0.6099 ± 0.0895,0.5258 ± 0.1469,0.5528 ± 0.1227,0.5514 ± 0.1549
train_roc_auc,0.9997 ± 0.0005,0.9955 ± 0.0035,0.9947 ± 0.0062,0.9974 ± 0.0040,0.9993 ± 0.0012
test_precision,0.5856 ± 0.1165,0.6340 ± 0.0554,0.5529 ± 0.0914,0.6014 ± 0.1016,0.5715 ± 0.1133
train_precision,0.9356 ± 0.0323,0.9112 ± 0.0274,0.9107 ± 0.0433,0.9297 ± 0.0324,0.9427 ± 0.0365
test_recall,0.7483 ± 0.1916,0.8456 ± 0.1120,0.6611 ± 0.1200,0.7183 ± 0.1875,0.7211 ± 0.1646
train_recall,1.0000 ± 0.0000,0.9909 ± 0.0124,0.9948 ± 0.0105,0.9987 ± 0.0056,1.0000 ± 0.0000
test_f1,0.6523 ± 0.1385,0.7197 ± 0.0540,0.5974 ± 0.0894,0.6452 ± 0.1293,0.6331 ± 0.1224
train_f1,0.9664 ± 0.0173,0.9491 ± 0.0151,0.9503 ± 0.0238,0.9627 ± 0.0181,0.9702 ± 0.0194


## ICANS >=2

In [10]:
y = targets["ae_summ_icans_highestgrade_v2"]

In [11]:
y = y.astype('float64')

In [12]:
y[y<2] = 0
y[y>=2] = 1


In [13]:
y.value_counts()

ae_summ_icans_highestgrade_v2
0.0    52
1.0    33
Name: count, dtype: int64

In [14]:
icans_2 = result_viewer(datasets, y, preprocessor)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5979 ± 0.1547,0.5999 ± 0.1384,0.5990 ± 0.1542,0.6499 ± 0.1209,0.6424 ± 0.1303
train_roc_auc,1.0000 ± 0.0000,0.9259 ± 0.0209,0.9977 ± 0.0024,0.9988 ± 0.0016,0.9987 ± 0.0018
test_precision,0.4981 ± 0.1659,0.4848 ± 0.2096,0.4731 ± 0.2144,0.5278 ± 0.1331,0.4932 ± 0.1417
train_precision,1.0000 ± 0.0000,0.8376 ± 0.0390,0.9718 ± 0.0233,0.9871 ± 0.0242,1.0000 ± 0.0000
test_recall,0.5214 ± 0.1712,0.4060 ± 0.1703,0.4357 ± 0.2062,0.5167 ± 0.1594,0.5869 ± 0.1493
train_recall,1.0000 ± 0.0000,0.7123 ± 0.0600,0.9734 ± 0.0297,0.9640 ± 0.0255,0.9472 ± 0.0295
test_f1,0.4947 ± 0.1436,0.4288 ± 0.1667,0.4426 ± 0.2015,0.5098 ± 0.1240,0.5279 ± 0.1283
train_f1,1.0000 ± 0.0000,0.7689 ± 0.0451,0.9724 ± 0.0231,0.9751 ± 0.0172,0.9726 ± 0.0158


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5025 ± 0.1341,0.5542 ± 0.0995,0.5080 ± 0.1336,0.5549 ± 0.1154,0.5160 ± 0.1371
train_roc_auc,0.9438 ± 0.0247,0.8757 ± 0.0358,0.9240 ± 0.0370,0.9278 ± 0.0192,0.9117 ± 0.0454
test_precision,0.3618 ± 0.1712,0.4159 ± 0.1936,0.3642 ± 0.1405,0.3988 ± 0.1996,0.4103 ± 0.1202
train_precision,0.9309 ± 0.0712,0.8293 ± 0.1015,0.8832 ± 0.0803,0.8918 ± 0.0787,0.8618 ± 0.0982
test_recall,0.3917 ± 0.2698,0.3976 ± 0.2221,0.3548 ± 0.2322,0.4357 ± 0.2496,0.4476 ± 0.1934
train_recall,0.8143 ± 0.1113,0.7194 ± 0.1687,0.7956 ± 0.1246,0.8104 ± 0.1200,0.8184 ± 0.1457
test_f1,0.3633 ± 0.2011,0.3750 ± 0.1573,0.3392 ± 0.1637,0.4011 ± 0.1998,0.4093 ± 0.1332
train_f1,0.8599 ± 0.0473,0.7475 ± 0.0769,0.8265 ± 0.0634,0.8384 ± 0.0475,0.8236 ± 0.0605


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4727 ± 0.1224,0.5608 ± 0.1334,0.4236 ± 0.0947,0.5822 ± 0.0953,0.4584 ± 0.1477
train_roc_auc,1.0000 ± 0.0002,0.9940 ± 0.0042,0.9990 ± 0.0017,0.9984 ± 0.0016,0.9999 ± 0.0006
test_precision,0.2496 ± 0.2320,0.3825 ± 0.2920,0.1842 ± 0.2169,0.5568 ± 0.3280,0.2331 ± 0.3234
train_precision,1.0000 ± 0.0000,0.9933 ± 0.0161,0.9962 ± 0.0168,0.9958 ± 0.0125,1.0000 ± 0.0000
test_recall,0.1440 ± 0.1379,0.1893 ± 0.1404,0.1167 ± 0.1408,0.2107 ± 0.1399,0.1048 ± 0.1254
train_recall,0.9429 ± 0.0415,0.7550 ± 0.0683,0.8826 ± 0.0460,0.8578 ± 0.0658,0.9467 ± 0.0410
test_f1,0.1762 ± 0.1607,0.2475 ± 0.1790,0.1407 ± 0.1665,0.2785 ± 0.1540,0.1325 ± 0.1581
train_f1,0.9701 ± 0.0226,0.8559 ± 0.0432,0.9352 ± 0.0257,0.9203 ± 0.0379,0.9722 ± 0.0220


# CRS 0,1
excluded

In [15]:
y = targets["ae_summ_crs_v2"]

In [16]:
y.value_counts()

ae_summ_crs_v2
1.0    79
0.0     6
Name: count, dtype: int64

In [17]:
crs_1 = result_viewer(datasets, y, preprocessor)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.2085 ± 0.1639,0.5760 ± 0.1758,0.2733 ± 0.2113,0.4256 ± 0.2174,0.4369 ± 0.2104
train_roc_auc,1.0000 ± 0.0000,0.9949 ± 0.0052,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_precision,0.9235 ± 0.0259,0.9257 ± 0.0260,0.9250 ± 0.0254,0.9258 ± 0.0258,0.9253 ± 0.0237
train_precision,1.0000 ± 0.0000,0.9599 ± 0.0123,1.0000 ± 0.0000,0.9945 ± 0.0074,0.9930 ± 0.0078
test_recall,0.9208 ± 0.0687,0.9523 ± 0.0567,0.9398 ± 0.0509,0.9523 ± 0.0526,0.9402 ± 0.0609
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9209 ± 0.0398,0.9380 ± 0.0350,0.9316 ± 0.0306,0.9382 ± 0.0330,0.9315 ± 0.0327
train_f1,1.0000 ± 0.0000,0.9795 ± 0.0065,1.0000 ± 0.0000,0.9973 ± 0.0037,0.9965 ± 0.0039


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5276 ± 0.1748,0.5181 ± 0.1993,0.4628 ± 0.0621,0.5594 ± 0.2039,0.4476 ± 0.0405
train_roc_auc,0.9950 ± 0.0218,0.9727 ± 0.0308,0.9342 ± 0.0853,0.9937 ± 0.0215,0.8526 ± 0.0701
test_precision,0.9325 ± 0.0379,0.9224 ± 0.0268,0.9237 ± 0.0238,0.9375 ± 0.0375,0.9216 ± 0.0286
train_precision,0.9992 ± 0.0034,0.9799 ± 0.0128,0.9838 ± 0.0123,0.9953 ± 0.0072,0.9746 ± 0.0120
test_recall,0.9052 ± 0.0617,0.9081 ± 0.0681,0.9181 ± 0.0657,0.9156 ± 0.0772,0.9046 ± 0.0840
train_recall,1.0000 ± 0.0000,0.9961 ± 0.0084,0.9992 ± 0.0035,0.9944 ± 0.0076,0.9969 ± 0.0062
test_f1,0.9171 ± 0.0357,0.9139 ± 0.0417,0.9195 ± 0.0349,0.9236 ± 0.0362,0.9113 ± 0.0522
train_f1,0.9996 ± 0.0017,0.9879 ± 0.0062,0.9914 ± 0.0065,0.9948 ± 0.0038,0.9856 ± 0.0056


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4244 ± 0.2339,0.5437 ± 0.1925,0.3956 ± 0.2935,0.5413 ± 0.2624,0.2852 ± 0.1625
train_roc_auc,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_precision,0.9290 ± 0.0243,0.9294 ± 0.0235,0.9292 ± 0.0235,0.9294 ± 0.0235,0.9294 ± 0.0235
train_precision,0.9830 ± 0.0117,0.9547 ± 0.0093,0.9776 ± 0.0090,0.9784 ± 0.0112,0.9702 ± 0.0120
test_recall,0.9967 ± 0.0145,1.0000 ± 0.0000,0.9969 ± 0.0136,1.0000 ± 0.0000,1.0000 ± 0.0000
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.9615 ± 0.0176,0.9633 ± 0.0129,0.9616 ± 0.0139,0.9633 ± 0.0129,0.9633 ± 0.0129
train_f1,0.9914 ± 0.0060,0.9768 ± 0.0048,0.9887 ± 0.0046,0.9891 ± 0.0057,0.9848 ± 0.0062


# CRS >=2

In [18]:
y = targets["ae_summ_highestgrade_v2"]
y = y.astype('float64')
y[y<2] = 0
y[y>=2] = 1

y.value_counts()

ae_summ_highestgrade_v2
0.0    55
1.0    30
Name: count, dtype: int64

In [19]:
crs_2 = result_viewer(datasets, y, preprocessor)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6318 ± 0.1060,0.5455 ± 0.1299,0.5636 ± 0.1250,0.6038 ± 0.1009,0.5508 ± 0.1164
train_roc_auc,1.0000 ± 0.0000,0.8972 ± 0.0240,0.9986 ± 0.0012,0.9991 ± 0.0012,1.0000 ± 0.0002
test_precision,0.4989 ± 0.2437,0.4030 ± 0.2316,0.4893 ± 0.1856,0.4696 ± 0.1584,0.4513 ± 0.2128
train_precision,1.0000 ± 0.0000,0.7621 ± 0.0481,0.9748 ± 0.0206,0.9836 ± 0.0238,0.9979 ± 0.0091
test_recall,0.4667 ± 0.2082,0.2917 ± 0.1656,0.4083 ± 0.1935,0.4250 ± 0.1622,0.3667 ± 0.1871
train_recall,1.0000 ± 0.0000,0.6188 ± 0.0662,0.9604 ± 0.0308,0.9688 ± 0.0291,0.9667 ± 0.0363
test_f1,0.4681 ± 0.2004,0.3163 ± 0.1573,0.4351 ± 0.1768,0.4354 ± 0.1440,0.3836 ± 0.1638
train_f1,1.0000 ± 0.0000,0.6815 ± 0.0526,0.9673 ± 0.0219,0.9758 ± 0.0202,0.9817 ± 0.0198


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4890 ± 0.1261,0.4761 ± 0.1189,0.5189 ± 0.1067,0.4716 ± 0.1434,0.5023 ± 0.1481
train_roc_auc,0.9401 ± 0.0386,0.8735 ± 0.0386,0.9042 ± 0.0550,0.9171 ± 0.0362,0.9295 ± 0.0461
test_precision,0.3212 ± 0.1548,0.2672 ± 0.1674,0.3529 ± 0.2110,0.3368 ± 0.1996,0.3723 ± 0.1896
train_precision,0.9305 ± 0.0652,0.8641 ± 0.0781,0.9018 ± 0.0830,0.8738 ± 0.1185,0.9278 ± 0.0694
test_recall,0.3333 ± 0.1900,0.2250 ± 0.1689,0.3083 ± 0.1991,0.3417 ± 0.1706,0.3250 ± 0.1935
train_recall,0.8396 ± 0.1181,0.6958 ± 0.1149,0.7937 ± 0.1137,0.8125 ± 0.1091,0.8063 ± 0.1433
test_f1,0.3176 ± 0.1571,0.2385 ± 0.1622,0.3065 ± 0.1626,0.3179 ± 0.1325,0.3282 ± 0.1584
train_f1,0.8762 ± 0.0732,0.7618 ± 0.0680,0.8338 ± 0.0487,0.8286 ± 0.0548,0.8527 ± 0.0870


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.6227 ± 0.1433,0.5621 ± 0.1395,0.6136 ± 0.1133,0.5439 ± 0.1025,0.6182 ± 0.1225
train_roc_auc,1.0000 ± 0.0002,0.9941 ± 0.0055,0.9966 ± 0.0034,0.9982 ± 0.0026,0.9995 ± 0.0010
test_precision,0.4321 ± 0.3402,0.1750 ± 0.2957,0.3852 ± 0.2651,0.4075 ± 0.3237,0.3937 ± 0.2876
train_precision,1.0000 ± 0.0000,0.9972 ± 0.0121,0.9977 ± 0.0099,1.0000 ± 0.0000,1.0000 ± 0.0000
test_recall,0.2000 ± 0.1354,0.0667 ± 0.1106,0.2250 ± 0.1516,0.1583 ± 0.1115,0.2167 ± 0.1590
train_recall,0.9437 ± 0.0401,0.6458 ± 0.0639,0.8417 ± 0.0408,0.7896 ± 0.0596,0.8958 ± 0.0519
test_f1,0.2507 ± 0.1555,0.0935 ± 0.1522,0.2610 ± 0.1448,0.2160 ± 0.1433,0.2587 ± 0.1648
train_f1,0.9706 ± 0.0214,0.7820 ± 0.0454,0.9125 ± 0.0235,0.8812 ± 0.0373,0.9443 ± 0.0294


# Treatment Response
CMR (Complete Response) vs others (PMR: Partial Response, SD: Stable Disease, PD: Progressive Disease)

In [20]:
y = targets["surv_bestresponse_car"]

In [21]:
y.value_counts()

surv_bestresponse_car
1.0    64
2.0    12
4.0     8
0.0     2
Name: count, dtype: int64

Early Death: 0, CMR: 1, PMR:2, SD: 3, PD:4

In [22]:
y = y.astype('int8')

In [23]:
y[y != 1] = 0

In [24]:
y.value_counts()

surv_bestresponse_car
1    64
0    22
Name: count, dtype: int64

In [25]:
outcome = result_viewer(datasets, y, preprocessor)

Results for Logistic Regression:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5386 ± 0.1522,0.6506 ± 0.1514,0.4954 ± 0.1350,0.6190 ± 0.1775,0.6328 ± 0.1835
train_roc_auc,1.0000 ± 0.0000,0.9531 ± 0.0193,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_precision,0.7516 ± 0.0714,0.7882 ± 0.0630,0.7693 ± 0.0631,0.7775 ± 0.0787,0.7934 ± 0.0787
train_precision,1.0000 ± 0.0000,0.8887 ± 0.0237,0.9811 ± 0.0155,1.0000 ± 0.0000,0.9981 ± 0.0057
test_recall,0.7462 ± 0.1015,0.8468 ± 0.1144,0.8170 ± 0.1180,0.7814 ± 0.1128,0.8051 ± 0.1274
train_recall,1.0000 ± 0.0000,0.9629 ± 0.0183,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.7454 ± 0.0707,0.8143 ± 0.0804,0.7887 ± 0.0782,0.7756 ± 0.0831,0.7918 ± 0.0745
train_f1,1.0000 ± 0.0000,0.9242 ± 0.0178,0.9904 ± 0.0079,1.0000 ± 0.0000,0.9990 ± 0.0029


Results for Decision Tree:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.4915 ± 0.1177,0.5223 ± 0.1214,0.4961 ± 0.1510,0.5236 ± 0.1183,0.5073 ± 0.1143
train_roc_auc,0.9518 ± 0.0325,0.8502 ± 0.0409,0.8915 ± 0.0593,0.8986 ± 0.0489,0.9400 ± 0.0362
test_precision,0.7377 ± 0.0550,0.7769 ± 0.0442,0.7584 ± 0.0693,0.7542 ± 0.0500,0.7521 ± 0.0701
train_precision,0.9445 ± 0.0332,0.8767 ± 0.0280,0.9082 ± 0.0256,0.9175 ± 0.0298,0.9341 ± 0.0386
test_recall,0.7663 ± 0.1019,0.8667 ± 0.0894,0.7958 ± 0.1128,0.8010 ± 0.1209,0.8013 ± 0.1007
train_recall,0.9854 ± 0.0212,0.9902 ± 0.0169,0.9883 ± 0.0168,0.9834 ± 0.0234,0.9727 ± 0.0335
test_f1,0.7483 ± 0.0659,0.8174 ± 0.0549,0.7734 ± 0.0780,0.7727 ± 0.0717,0.7701 ± 0.0523
train_f1,0.9639 ± 0.0152,0.9296 ± 0.0140,0.9464 ± 0.0181,0.9488 ± 0.0164,0.9519 ± 0.0164


Results for Random Forest:


,Patients,Clinical,Clinical_A,Clinical_B,Clinical_Delta
test_roc_auc,0.5383 ± 0.1909,0.6277 ± 0.1422,0.5221 ± 0.1800,0.5590 ± 0.1828,0.5876 ± 0.1822
train_roc_auc,0.9994 ± 0.0012,0.9950 ± 0.0051,0.9960 ± 0.0054,0.9986 ± 0.0017,1.0000 ± 0.0000
test_precision,0.7394 ± 0.0356,0.7480 ± 0.0271,0.7332 ± 0.0342,0.7626 ± 0.0525,0.7395 ± 0.0257
train_precision,0.9231 ± 0.0230,0.8416 ± 0.0257,0.8958 ± 0.0243,0.8772 ± 0.0192,0.9247 ± 0.0212
test_recall,0.9327 ± 0.0732,0.9843 ± 0.0397,0.9375 ± 0.0916,0.9718 ± 0.0526,0.9654 ± 0.0569
train_recall,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000
test_f1,0.8240 ± 0.0455,0.8496 ± 0.0267,0.8213 ± 0.0507,0.8538 ± 0.0471,0.8367 ± 0.0314
train_f1,0.9599 ± 0.0125,0.9138 ± 0.0152,0.9449 ± 0.0136,0.9345 ± 0.0109,0.9608 ± 0.0115


All the models in all the classification problems are hugely overfit, which is expected when we have fewer data points than features. This problem will be the focus of the next steps.